# 全行业 PCA：训练选参数 → 验证选行业

先运行 `sector_pca_stat_arb.ipynb` 导出各行业 CSV，再从头运行本文件。每个行业只按训练数据筛选 ETF、搜索超参数，固定训练最优参数后比较验证综合得分。测试收益不在这里计算。

沿用对数收益、PCA 残差 AR(1)、对称开平仓带、因子对冲和波动率调整；费用用权重差估算，固定初始资金加法累计。仍假设信号日收盘成交，不计借券费或滑点。训练排名是样本内结果，验证排名属于行业选择。

综合得分 = Min-Max 净值得分 + Min-Max 夏普得分 + Min-Max 回撤得分，三个指标等权相加。训练时在每个行业的有效参数组合内归一化；验证时在有效行业候选间重新归一化，不使用测试数据。最大回撤保存为负数，越接近 0 得分越高。

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json

# 修改日期后从头运行，三个时间段互不重叠
train_start = pd.Timestamp('2023-02-02')
train_end = pd.Timestamp('2025-03-31')
valid_start = pd.Timestamp('2025-04-01')
valid_end = pd.Timestamp('2025-12-31')
test_start = pd.Timestamp('2026-01-01')
test_end = pd.Timestamp('2026-09-30')

# 保留当前网格；最长窗口为 252 个交易日
window_values = [40, 60, 80, 100, 120, 150, 180, 200, 252]
k_values = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
s_open_values = [0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5]
s_close_values = [0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5]
fee = 0.0005
hl_min, hl_max = 5, 60
# 固定使用三个指标归一化后的综合得分，不设置主观权重
selection_metric = '综合得分'

input_dir = Path('data/etf_by_sector')
output_dir = Path('outputs/pca_selection')
selected_dir = Path('data/selected_sector')
output_dir.mkdir(parents=True, exist_ok=True)
sector_files = sorted(input_dir.glob('*.csv'))

# 用同一交易日历，所有窗口、行业从同一日开始比较训练收益
calendar = pd.DatetimeIndex(pd.read_csv(sector_files[0], usecols=['date'], parse_dates=['date'])['date'].unique()).sort_values()
train_dates = calendar[(calendar >= train_start) & (calendar <= train_end)]
train_backtest_start = train_dates[max(window_values)]
backtest_dates = train_dates[train_dates >= train_backtest_start]
if not (train_start <= train_end < valid_start <= valid_end < test_start <= test_end):
    raise ValueError('请按时间顺序设置互不重叠的训练、验证、测试日期')
if max(window_values) > 252:
    raise ValueError('最长 window 不能超过 252')
print(f'训练有效回测：{backtest_dates[0].date()}～{backtest_dates[-1].date()}，{len(backtest_dates)} 日')
print(f'待搜索行业：{len(sector_files)}；训练和验证均按净值、夏普、回撤的归一化综合得分选择')

训练有效回测：2024-02-08～2025-03-31，272 日
待搜索行业：12；训练和验证均按净值、夏普、回撤的归一化综合得分选择


## 1. 每个行业只在训练集选择一组最优参数

ETF 名单仅由训练期确定。不同 window 和 k 不重复计算开平仓阈值之外的模型，同一模型结果供多个阈值使用。主成分数必须小于 ETF 数及窗口长度。

网格期间在内存比较，避免导出几万份临时文件。每个行业只保存训练最优权重，然后读取 CSV 复核。交易活跃天数也列在结果中，便于识别仅凭少数交易日得到的高排名；目前不额外设置活跃度门槛。

先完成全部组合的指标表，再归一化评分；不能在循环中提前判断综合最优。任一指标非有限（例如没有交易、夏普为 NaN）的组合保留在表内，但不参加评分。某指标全相同时统一给 0.5 分，不影响排名。

In [ ]:
train_tables = []
train_winners = []
sector_codes = {}

for sector_file in sector_files:
    sector = sector_file.stem
    sector_dir = output_dir / sector
    sector_dir.mkdir(parents=True, exist_ok=True)
    data = pd.read_csv(sector_file, dtype={'code': str}, parse_dates=['date'])
    # 测试期不参与筛选、模型拟合或参数比较
    train = data.loc[data['date'].between(train_start, train_end)].copy()
    candidates = train.loc[
        train['date'].eq(train_dates[0]) & train['fund_type'].eq('股票型'), ['code', 'name']
    ].sort_values('code')
    train_returns = train.pivot(index='date', columns='code', values='log_ret_1d').reindex(train_dates)[candidates['code']]
    train_prices = train.pivot(index='date', columns='code', values='close_hfq').reindex(train_dates)[candidates['code']]
    # 仅保留训练期间完整、有限、价格为正且波动非零的 ETF
    eligible = (np.isfinite(train_returns).all() & np.isfinite(train_prices).all()
                & train_prices.gt(0).all() & train_returns.std().gt(0))
    codes = eligible.index[eligible].tolist()
    sector_codes[sector] = codes
    candidates.loc[candidates['code'].isin(codes)].to_csv(sector_dir / 'etfs.csv', index=False)
    returns = train_returns[codes]
    asset_returns = np.expm1(returns.loc[backtest_dates])
    trials = []
    for window in window_values:
        for k in k_values:
            if k >= min(len(codes), window):
                continue
            scores = pd.DataFrame(np.nan, index=returns.index, columns=returns.columns)
            half_lives, vols = scores.copy(), scores.copy()
            loadings = {}
            for i in range(window - 1, len(returns)):
                sample = returns.iloc[i-window+1:i+1].to_numpy() # shape(w, n),n是ETF数量
                std = sample.std(axis=0)# 对windows内取波动率
                standardized = (sample - sample.mean(axis=0)) / std # 归一化，方便用主成分
                _, _, vt = np.linalg.svd(standardized, full_matrices=False) # 得到的是vt矩阵，shape(n, n)
                loading = vt[:k] # k是主成分数量 shape(k, n)
                residual = standardized - (standardized @ loading.T) @ loading # 先将原来的n个etf收益转换为k个主成分因子，然后再利用主成分重构原始数据，最后计算残差，shape(w,n)
                # 这里的残差是剥除掉波动率最高的k个主成分因子后的残差，得到的是每个etf的各自的特点
                X = residual.cumsum(axis=0) # 按照window内对残差求和，shape(n)
                lag, current = X[:-1], X[1:] # 取前w-1个etf的残差，当前etf的残差
                centered = lag - lag.mean(axis=0)
                # 拟合ar(1)方程 x_t = a + b * x_t-1 + e_t
                b = (centered * (current - current.mean(axis=0))).sum(axis=0) / (centered**2).sum(axis=0) # 直接计算一阶滞后相关系数
                a = current.mean(axis=0) - b * lag.mean(axis=0)
                innovation = current - a - b * lag
                b = np.where((b > 0) & (b < 1), b, np.nan)
                m = a / (1 - b) # 假设x_t-1的期望= x_t的期望，也就是期望不变的前提下
                sigma_eq = innovation.std(axis=0, ddof=2) / np.sqrt(1 - b**2) # 当前累计残差偏离均值多少个标准差
                date = returns.index[i]
                scores.loc[date] = (X[-1] - m) / sigma_eq # 计算z-score
                half_lives.loc[date] = -np.log(2) / np.log(b)
                vols.loc[date] = std
                loadings[date] = loading
            score_values = scores.to_numpy()
            hl_values = half_lives.to_numpy()
            vol_values = vols.to_numpy()
            for s_open in s_open_values:
                for s_close in s_close_values:
                    if s_close >= s_open:
                        continue
                    direction = np.zeros(len(codes))
                    weight_values = np.zeros(scores.shape)
                    for i in range(window - 1, len(returns)):
                        s, hl = score_values[i], hl_values[i]
                        valid = np.isfinite(s) & (hl >= hl_min) & (hl <= hl_max)
                        direction = np.where((np.abs(s) < s_close) | ~valid, 0, direction)
                        direction = np.where(valid & (s < -s_open), 1, direction)
                        direction = np.where(valid & (s > s_open), -1, direction)
                        loading = loadings[returns.index[i]]
                        weight = (direction - loading.T @ (loading @ direction)) / vol_values[i]
                        weight_values[i] = weight / max(np.abs(weight).sum(), 1e-12)
                    signal_weights = pd.DataFrame(weight_values, index=returns.index, columns=codes)
                    weights = signal_weights.shift(1).loc[backtest_dates]
                    turnover = weights.diff().fillna(weights).abs().sum(axis=1)
                    ret = (weights * asset_returns).sum(axis=1) - turnover * fee
                    nav = np.r_[1.0, 1 + ret.cumsum()]
                    sharpe = ret.mean() / ret.std(ddof=0) * np.sqrt(252) if ret.std(ddof=0) > 0 else np.nan
                    row = {'行业': sector, 'window': window, 'k': k, 's_open': s_open, 's_close': s_close,
                           '年化夏普': sharpe, '期末净值': nav[-1], '累计收益': nav[-1] - 1,
                           '最大回撤': (nav / np.maximum.accumulate(nav) - 1).min(),
                           '累计换手': turnover.sum(), '交易费用': turnover.sum() * fee,
                           '持仓天数': int(weights.abs().sum(axis=1).gt(1e-12).sum()), '回测天数': len(ret)}
                    trials.append(row)
            print(f'{sector}：window={window}, k={k} 完成，已比较 {len(trials)} 组', flush=True)
    table = pd.DataFrame(trials)
    if table.empty:
        raise ValueError(f'{sector} 没有可用组合，请检查 ETF 数量、网格和训练区间')
    # 只在本行业的有效训练组合中计算归一化范围，不混入其他行业或验证数据
    metric_columns = ['期末净值', '年化夏普', '最大回撤']
    score_columns = ['净值得分', '夏普得分', '回撤得分']
    table['可评分'] = np.isfinite(table[metric_columns]).all(axis=1)
    values = table.loc[table['可评分'], metric_columns]
    if values.empty:
        raise ValueError(f'{sector} 没有三个指标均有效的组合，请检查交易活跃度')
    span = values.max() - values.min()
    # 回撤为负数，越接近 0 越好，因此三个指标的方向一致
    normalized = (values - values.min()) / span.replace(0, np.nan)
    # 某一指标全部相同，统一给 0.5 分；不把无效组合填成可用分数
    table.loc[table['可评分'], score_columns] = normalized.fillna(0.5).to_numpy()
    # min_count=3：必须三个分数都有效，避免 NaN 被 sum 当成 0
    table['综合得分'] = table[score_columns].sum(axis=1, min_count=3)
    table = table.sort_values('综合得分', ascending=False, kind='stable', na_position='last').reset_index(drop=True)
    table.to_csv(sector_dir / 'train_grid.csv', index=False, encoding='utf-8-sig')
    train_tables.append(table)
    train_winners.append(table.iloc[0].to_dict())
    # 完整评分后才知道赢家，只重算这一组训练信号，不保存全部组合的权重
    best = table.iloc[0]
    window, k = int(best['window']), int(best['k'])
    s_open, s_close = float(best['s_open']), float(best['s_close'])
    scores = pd.DataFrame(np.nan, index=returns.index, columns=returns.columns)
    half_lives, vols = scores.copy(), scores.copy()
    loadings = {}
    for i in range(window - 1, len(returns)):
        sample = returns.iloc[i-window+1:i+1].to_numpy() # shape(w, n),n是ETF数量
        std = sample.std(axis=0)# 对windows内取波动率
        standardized = (sample - sample.mean(axis=0)) / std # 归一化，方便用主成分
        _, _, vt = np.linalg.svd(standardized, full_matrices=False) # 得到的是vt矩阵，shape(n, n)
        loading = vt[:k] # k是主成分数量 shape(k, n)
        residual = standardized - (standardized @ loading.T) @ loading # 先将原来的n个etf收益转换为k个主成分因子，然后再利用主成分重构原始数据，最后计算残差，shape(w,n)
        # 这里的残差是剥除掉波动率最高的k个主成分因子后的残差，得到的是每个etf的各自的特点
        X = residual.cumsum(axis=0) # 按照window内对残差求和，shape(n)
        lag, current = X[:-1], X[1:] # 取前w-1个etf的残差，当前etf的残差
        centered = lag - lag.mean(axis=0)
        # 拟合ar(1)方程 x_t = a + b * x_t-1 + e_t
        b = (centered * (current - current.mean(axis=0))).sum(axis=0) / (centered**2).sum(axis=0) # 直接计算一阶滞后相关系数
        a = current.mean(axis=0) - b * lag.mean(axis=0)
        innovation = current - a - b * lag
        b = np.where((b > 0) & (b < 1), b, np.nan)
        m = a / (1 - b) # 假设x_t-1的期望= x_t的期望，也就是期望不变的前提下
        sigma_eq = innovation.std(axis=0, ddof=2) / np.sqrt(1 - b**2) # 当前累计残差偏离均值多少个标准差
        date = returns.index[i]
        scores.loc[date] = (X[-1] - m) / sigma_eq # 计算z-score
        half_lives.loc[date] = -np.log(2) / np.log(b)
        vols.loc[date] = std
        loadings[date] = loading
    # 先建日期 × ETF 的权重表，初始全部空仓
    weights = pd.DataFrame(0.0, index=scores.index, columns=scores.columns)
    # 记录每只 ETF 的残差交易方向：1 做多，-1 做空，0 空仓
    direction = np.zeros(len(scores.columns))
    # 按日期更新方向，未触发开平仓条件时保留原方向
    for date, loading in loadings.items():
        # 取当天每只 ETF 的 s 信号
        s = scores.loc[date].to_numpy()
        # 取当天估计的均值回归半衰期，单位是交易日
        hl = half_lives.loc[date].to_numpy()
        # 只使用有效信号，并要求半衰期在顶部设定的范围内
        valid = np.isfinite(s) & (hl >= hl_min) & (hl <= hl_max)
        # 进入平仓带或信号无效时，把主动方向清零
        direction = np.where((np.abs(s) < s_close) | ~valid, 0, direction)
        # 残差偏低时做多，等它向均值回归
        direction = np.where(valid & (s < -s_open), 1, direction)
        # 残差偏高时做空，等它向均值回归
        direction = np.where(valid & (s > s_open), -1, direction)
        # 扣掉方向在 PCA 因子上的投影，再除以波动率，转成 ETF 对冲权重，这里主要是计算仓位强度
        weight = (direction - loading.T @ (loading @ direction)) / vols.loc[date].to_numpy() # loading.shape = (k, n), direction.shape = (n,)
        # 按绝对权重之和归一化；没有仓位时保持为 0，避免除以 0
        weights.loc[date] = weight / max(np.abs(weight).sum(), 1e-12)
    best_weights = weights.iloc[window-1:].copy()
    # 保存训练最优信号，再读取计算明细；与网格保持相同账户起点
    best_weights.to_csv(sector_dir / 'train_weights.csv', index_label='date')
    signal_weights = pd.read_csv(sector_dir / 'train_weights.csv', parse_dates=['date']).set_index('date')
    weights = signal_weights.shift(1).loc[backtest_dates]
    turnover = weights.diff().fillna(weights).abs().sum(axis=1)
    result = pd.DataFrame(index=backtest_dates)
    result['gross_ret'] = (weights * asset_returns).sum(axis=1)
    result['turnover'] = turnover
    result['trade_cost'] = turnover * fee
    result['strategy_ret'] = result['gross_ret'] - result['trade_cost']
    result['net_value'] = 1 + result['strategy_ret'].cumsum()
    result.to_csv(sector_dir / 'train_backtest.csv', index_label='date')
    print(f'{sector} 训练选参完成：{len(codes)} 只 ETF，{len(table)} 组', flush=True)

军工：window=40, k=1 完成，已比较 55 组
军工：window=40, k=2 完成，已比较 110 组
军工：window=40, k=3 完成，已比较 165 组
军工：window=40, k=4 完成，已比较 220 组
军工：window=40, k=5 完成，已比较 275 组
军工：window=60, k=1 完成，已比较 330 组
军工：window=60, k=2 完成，已比较 385 组
军工：window=60, k=3 完成，已比较 440 组
军工：window=60, k=4 完成，已比较 495 组
军工：window=60, k=5 完成，已比较 550 组
军工：window=80, k=1 完成，已比较 605 组
军工：window=80, k=2 完成，已比较 660 组
军工：window=80, k=3 完成，已比较 715 组
军工：window=80, k=4 完成，已比较 770 组
军工：window=80, k=5 完成，已比较 825 组
军工：window=100, k=1 完成，已比较 880 组
军工：window=100, k=2 完成，已比较 935 组
军工：window=100, k=3 完成，已比较 990 组
军工：window=100, k=4 完成，已比较 1045 组
军工：window=100, k=5 完成，已比较 1100 组
军工：window=120, k=1 完成，已比较 1155 组
军工：window=120, k=2 完成，已比较 1210 组
军工：window=120, k=3 完成，已比较 1265 组
军工：window=120, k=4 完成，已比较 1320 组
军工：window=120, k=5 完成，已比较 1375 组
军工：window=150, k=1 完成，已比较 1430 组
军工：window=150, k=2 完成，已比较 1485 组
军工：window=150, k=3 完成，已比较 1540 组
军工：window=150, k=4 完成，已比较 1595 组
军工：window=150, k=5 完成，已比较 1650 组
军工：window=180, k=1 完成，已比较 1705 组
军工：window=1

## 2. 查看所有训练参数表及各行业训练最优参数

每张 train_grid.csv 和 all_train_grid.csv 保留原始净值、夏普、最大回撤，以及三个归一化分数、综合得分和可评分标记。综合得分跨行业的训练表不直接比较；选行业只看验证阶段的重新评分。

In [ ]:
df_grid = pd.concat(train_tables, ignore_index=True)
df_grid.to_csv(output_dir / 'all_train_grid.csv', index=False, encoding='utf-8-sig')
df_train_best = pd.DataFrame(train_winners)
df_train_best.to_csv(output_dir / 'train_best_by_sector.csv', index=False, encoding='utf-8-sig')
df_train_best

,行业,window,k,s_open,s_close,年化夏普,期末净值,累计收益,最大回撤,累计换手,交易费用,持仓天数,回测天数,可评分,净值得分,夏普得分,回撤得分,综合得分
0,军工,252,1,0.75,0.10,2.060575,1.061307,0.061307,-0.009321,34.097589,0.017049,228,272,True,1.000000,0.956879,0.874405,2.831285
1,农业,200,2,0.75,0.25,2.106306,1.054368,0.054368,-0.011184,55.131777,0.027566,201,272,True,0.940208,0.978451,0.892144,2.810804
2,医药健康,252,2,2.50,0.50,1.710061,1.063090,0.063090,-0.011423,8.125682,0.004063,103,272,True,0.909561,0.949389,0.916140,2.775090
3,周期材料,60,1,1.50,0.25,2.948304,1.181746,0.181746,-0.031249,68.046480,0.034023,176,272,True,0.960925,0.984575,0.783217,2.728717
4,房地产基建,252,2,0.75,0.50,1.098118,1.960984,0.960984,-0.025506,33.692615,0.016846,247,272,True,1.000000,0.699817,0.962770,2.662587
5,新能源与环保,252,14,0.50,0.10,0.927496,1.681626,0.681626,-0.010573,110.442195,0.055221,219,272,True,1.000000,0.862642,0.989350,2.851992
6,汽车新能源车,100,1,1.00,0.75,3.593784,1.208740,0.208740,-0.017324,103.512876,0.051756,201,272,True,1.000000,1.000000,0.831470,2.831470
7,消费,252,3,1.25,0.75,3.242293,1.135104,0.135104,-0.015329,55.146041,0.027573,272,272,True,0.905866,0.993198,0.924674,2.823738
8,科技TMT,200,1,1.00,0.75,1.942723,1.180003,0.180003,-0.048598,70.027917,0.035014,272,272,True,0.943991,0.967542,0.799448,2.710980
9,能源,100,1,2.50,0.10,1.033727,1.241109,0.241109,-0.017081,2.643486,0.001322,79,272,True,1.000000,0.800411,0.951271,2.751682


## 3. 固定各行业训练最优参数，分别运行验证集

每个行业仅评估训练阶段选出的一组参数，不在验证集再次搜索。验证收益从 valid_start 后的首个交易日开始，向前取该行业 window 日预热，保留首日前一天的信号；方向状态与账户重新开始，不继承训练期持仓。历史窗口可以使用训练数据，不使用未来数据。

In [ ]:
validation_rows = []
for _, best in df_train_best.iterrows():
    sector = best['行业']
    window, k = int(best['window']), int(best['k'])
    s_open, s_close = float(best['s_open']), float(best['s_close'])
    codes = sector_codes[sector]
    sector_dir = output_dir / sector
    data = pd.read_csv(input_dir / f'{sector}.csv', dtype={'code': str}, parse_dates=['date'])
    # 先截断到验证末日，不把测试行送进滚动模型
    data = data.loc[data['date'].between(train_start, valid_end)]
    dates = pd.DatetimeIndex(data['date'].unique()).sort_values()
    i = dates.searchsorted(valid_start)
    valid_first = dates[i]
    valid_data = data.loc[data['date'] >= dates[i-window]]
    returns = valid_data.pivot(index='date', columns='code', values='log_ret_1d')[codes]
    scores = pd.DataFrame(np.nan, index=returns.index, columns=returns.columns)
    half_lives, vols = scores.copy(), scores.copy()
    loadings = {}
    for i in range(window - 1, len(returns)):
        sample = returns.iloc[i-window+1:i+1].to_numpy() # shape(w, n),n是ETF数量
        std = sample.std(axis=0)# 对windows内取波动率
        standardized = (sample - sample.mean(axis=0)) / std # 归一化，方便用主成分
        _, _, vt = np.linalg.svd(standardized, full_matrices=False) # 得到的是vt矩阵，shape(n, n)
        loading = vt[:k] # k是主成分数量 shape(k, n)
        residual = standardized - (standardized @ loading.T) @ loading # 先将原来的n个etf收益转换为k个主成分因子，然后再利用主成分重构原始数据，最后计算残差，shape(w,n)
        # 这里的残差是剥除掉波动率最高的k个主成分因子后的残差，得到的是每个etf的各自的特点
        X = residual.cumsum(axis=0) # 按照window内对残差求和，shape(n)
        lag, current = X[:-1], X[1:] # 取前w-1个etf的残差，当前etf的残差
        centered = lag - lag.mean(axis=0)
        # 拟合ar(1)方程 x_t = a + b * x_t-1 + e_t
        b = (centered * (current - current.mean(axis=0))).sum(axis=0) / (centered**2).sum(axis=0) # 直接计算一阶滞后相关系数
        a = current.mean(axis=0) - b * lag.mean(axis=0)
        innovation = current - a - b * lag
        b = np.where((b > 0) & (b < 1), b, np.nan)
        m = a / (1 - b) # 假设x_t-1的期望= x_t的期望，也就是期望不变的前提下
        sigma_eq = innovation.std(axis=0, ddof=2) / np.sqrt(1 - b**2) # 当前累计残差偏离均值多少个标准差
        date = returns.index[i]
        scores.loc[date] = (X[-1] - m) / sigma_eq # 计算z-score
        half_lives.loc[date] = -np.log(2) / np.log(b)
        vols.loc[date] = std
        loadings[date] = loading
    # 先建日期 × ETF 的权重表，初始全部空仓
    weights = pd.DataFrame(0.0, index=scores.index, columns=scores.columns)
    # 记录每只 ETF 的残差交易方向：1 做多，-1 做空，0 空仓
    direction = np.zeros(len(scores.columns))
    # 按日期更新方向，未触发开平仓条件时保留原方向
    for date, loading in loadings.items():
        # 取当天每只 ETF 的 s 信号
        s = scores.loc[date].to_numpy()
        # 取当天估计的均值回归半衰期，单位是交易日
        hl = half_lives.loc[date].to_numpy()
        # 只使用有效信号，并要求半衰期在顶部设定的范围内
        valid = np.isfinite(s) & (hl >= hl_min) & (hl <= hl_max)
        # 进入平仓带或信号无效时，把主动方向清零
        direction = np.where((np.abs(s) < s_close) | ~valid, 0, direction)
        # 残差偏低时做多，等它向均值回归
        direction = np.where(valid & (s < -s_open), 1, direction)
        # 残差偏高时做空，等它向均值回归
        direction = np.where(valid & (s > s_open), -1, direction)
        # 扣掉方向在 PCA 因子上的投影，再除以波动率，转成 ETF 对冲权重，这里主要是计算仓位强度
        weight = (direction - loading.T @ (loading @ direction)) / vols.loc[date].to_numpy() # loading.shape = (k, n), direction.shape = (n,)
        # 按绝对权重之和归一化；没有仓位时保持为 0，避免除以 0
        weights.loc[date] = weight / max(np.abs(weight).sum(), 1e-12)

    scores.iloc[window-1:].to_csv(sector_dir / 'valid_scores.csv', index_label='date')
    weights.iloc[window-1:].to_csv(sector_dir / 'valid_weights.csv', index_label='date')
    signal_weights = pd.read_csv(sector_dir / 'valid_weights.csv', parse_dates=['date']).set_index('date')
    weights = signal_weights.shift(1).loc[valid_first:valid_end]
    asset_returns = np.expm1(returns.loc[weights.index])
    turnover = weights.diff().fillna(weights).abs().sum(axis=1)
    result = pd.DataFrame(index=weights.index)
    result['gross_ret'] = (weights * asset_returns).sum(axis=1)
    result['turnover'] = turnover
    result['trade_cost'] = turnover * fee
    result['strategy_ret'] = result['gross_ret'] - result['trade_cost']
    result['net_value'] = 1 + result['strategy_ret'].cumsum()
    result.to_csv(sector_dir / 'valid_backtest.csv', index_label='date')
    ret = result['strategy_ret']
    nav = np.r_[1.0, result['net_value']]
    validation_rows.append({
        '行业': sector, 'window': window, 'k': k, 's_open': s_open, 's_close': s_close,
        '训练年化夏普': float(best['年化夏普']), '训练期末净值': float(best['期末净值']),
        '训练最大回撤': float(best['最大回撤']), '训练综合得分': float(best['综合得分']),
        '验证年化夏普': ret.mean() / ret.std(ddof=0) * np.sqrt(252) if ret.std(ddof=0) > 0 else np.nan,
        '验证期末净值': nav[-1], '验证累计收益': nav[-1] - 1,
        '验证最大回撤': (nav / np.maximum.accumulate(nav) - 1).min(),
        '验证持仓天数': int(weights.abs().sum(axis=1).gt(1e-12).sum()), '验证天数': len(ret)
    })
    print(f'{sector}：验证期末净值 {nav[-1]:.6f}', flush=True)

军工：验证期末净值 0.953052
农业：验证期末净值 0.962192
医药健康：验证期末净值 1.007978
周期材料：验证期末净值 1.083631
房地产基建：验证期末净值 0.973580
新能源与环保：验证期末净值 0.965487
汽车新能源车：验证期末净值 0.950093
消费：验证期末净值 0.996663
科技TMT：验证期末净值 0.882785
能源：验证期末净值 1.001558
金融：验证期末净值 0.989627
高端制造工业：验证期末净值 1.087422


## 4. 验证指标归一化，按综合得分给行业排名

只在各行业训练最优参数的验证结果间重新计算 Min-Max 范围。净值、夏普、负数最大回撤都越大越好，三个得分直接相加。某指标全部相同统一给 0.5 分；任一指标无效的行业保留在表内，但不给综合得分和排名。并列时保留行业文件名顺序。测试指标不参与计算。

In [ ]:
df_validation = pd.DataFrame(validation_rows)
metric_columns = ['验证期末净值', '验证年化夏普', '验证最大回撤']
score_columns = ['验证净值得分', '验证夏普得分', '验证回撤得分']
df_validation['可评分'] = np.isfinite(df_validation[metric_columns]).all(axis=1)
values = df_validation.loc[df_validation['可评分'], metric_columns]
if values.empty:
    raise ValueError('没有三个验证指标均有效的行业，无法选出赢家')
span = values.max() - values.min()
normalized = (values - values.min()) / span.replace(0, np.nan)
df_validation.loc[df_validation['可评分'], score_columns] = normalized.fillna(0.5).to_numpy()
df_validation['验证综合得分'] = df_validation[score_columns].sum(axis=1, min_count=3)
df_validation = df_validation.sort_values('验证综合得分', ascending=False, kind='stable', na_position='last').reset_index(drop=True)
# 无效行业没有排名；有效行业并列时按照现有文件名顺序排位
df_validation.insert(0, '验证排名', df_validation['验证综合得分'].rank(ascending=False, method='first').astype('Int64'))
df_validation.to_csv(output_dir / 'validation_ranking.csv', index=False, encoding='utf-8-sig')
df_validation

,验证排名,行业,window,k,s_open,s_close,训练年化夏普,训练期末净值,训练最大回撤,训练综合得分,...,验证期末净值,验证累计收益,验证最大回撤,验证持仓天数,验证天数,可评分,验证净值得分,验证夏普得分,验证回撤得分,验证综合得分
0,1,周期材料,60,1,1.50,0.25,2.948304,1.181746,-0.031249,2.728717,...,1.083631,0.083631,-0.049539,125,186,True,0.981473,1.000000,0.664616,2.646089
1,2,医药健康,252,2,2.50,0.50,1.710061,1.063090,-0.011423,2.775090,...,1.007978,0.007978,-0.004418,30,186,True,0.611783,0.950650,1.000000,2.562433
2,3,高端制造工业,100,1,0.75,0.50,1.903548,1.165283,-0.038427,2.670202,...,1.087422,0.087422,-0.090671,169,186,True,1.000000,0.847787,0.358891,2.206678
3,4,能源,100,1,2.50,0.10,1.033727,1.241109,-0.017081,2.751682,...,1.001558,0.001558,-0.037158,29,186,True,0.580407,0.755240,0.756642,2.092289
4,5,消费,252,3,1.25,0.75,3.242293,1.135104,-0.015329,2.823738,...,0.996663,-0.003337,-0.033521,186,186,True,0.556489,0.723595,0.783679,2.063763
5,6,金融,252,2,2.50,0.50,2.401802,1.051948,-0.006259,2.758803,...,0.989627,-0.010373,-0.013055,65,186,True,0.522104,0.427619,0.935796,1.885519
6,7,军工,252,1,0.75,0.10,2.060575,1.061307,-0.009321,2.831285,...,0.953052,-0.046948,-0.073013,185,186,True,0.343376,0.595201,0.490136,1.428714
7,8,房地产基建,252,2,0.75,0.50,1.098118,1.960984,-0.025506,2.662587,...,0.973580,-0.026420,-0.098219,147,186,True,0.443688,0.661536,0.302784,1.408007
8,9,农业,200,2,0.75,0.25,2.106306,1.054368,-0.011184,2.810804,...,0.962192,-0.037808,-0.041612,163,186,True,0.388037,0.244324,0.723537,1.355898
9,10,汽车新能源车,100,1,1.00,0.75,3.593784,1.208740,-0.017324,2.831470,...,0.950093,-0.049907,-0.087340,147,186,True,0.328914,0.557466,0.383646,1.270026


## 5. 把验证综合得分第一名的数据与参数保存到 data/selected_sector

这里是在行业和参数选择完成后，打包该行业已入选 ETF 的完整原始数据，供最后一个 Notebook 回测。保存测试行不等于使用测试表现选行业；本文件不计算测试收益。每次完整运行成功后更新 etf.csv 和 params.json。

In [ ]:
winner = df_validation.iloc[0]
sector = winner['行业']
codes = sector_codes[sector]
selected_data = pd.read_csv(input_dir / f'{sector}.csv', dtype={'code': str}, parse_dates=['date'])
selected_data = selected_data.loc[selected_data['code'].isin(codes) & selected_data['date'].between(train_start, test_end)].copy()
selected_dir.mkdir(parents=True, exist_ok=True)
selected_data.to_csv(selected_dir / 'etf.csv', index=False, encoding='utf-8-sig')
best_params = {
    'sector': sector, 'window': int(winner['window']), 'k': int(winner['k']),
    's_open': float(winner['s_open']), 's_close': float(winner['s_close']), 'fee': fee,
    'half_life_min': hl_min, 'half_life_max': hl_max,
    'train_selection_metric': selection_metric, 'sector_selection_metric': '验证综合得分',
    'normalization': 'min_max', 'score_rule': '净值得分 + 夏普得分 + 回撤得分（等权）',
    'train_start': str(train_start.date()), 'train_end': str(train_end.date()),
    'valid_start': str(valid_start.date()), 'valid_end': str(valid_end.date()),
    'test_start': str(test_start.date()), 'test_end': str(test_end.date()),
    'train_backtest_start': str(train_backtest_start.date()), 'eligible_codes': codes,
    'train_sharpe': float(winner['训练年化夏普']), 'train_final_nav': float(winner['训练期末净值']),
    'valid_sharpe': float(winner['验证年化夏普']), 'valid_final_nav': float(winner['验证期末净值']),
    'train_max_drawdown': float(winner['训练最大回撤']), 'valid_max_drawdown': float(winner['验证最大回撤']),
    'train_composite_score': float(winner['训练综合得分']), 'valid_composite_score': float(winner['验证综合得分'])
}
# 没有持仓时夏普可能为 NaN，JSON 中用 null 表示
best_params = {key: None if isinstance(value, float) and not np.isfinite(value) else value for key, value in best_params.items()}
(selected_dir / 'params.json').write_text(json.dumps(best_params, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
best_params

{'sector': '周期材料',
 'window': 60,
 'k': 1,
 's_open': 1.5,
 's_close': 0.25,
 'fee': 0.0005,
 'half_life_min': 5,
 'half_life_max': 60,
 'train_selection_metric': '综合得分',
 'sector_selection_metric': '验证综合得分',
 'normalization': 'min_max',
 'score_rule': '净值得分 + 夏普得分 + 回撤得分（等权）',
 'train_start': '2023-02-02',
 'train_end': '2025-03-31',
 'valid_start': '2025-04-01',
 'valid_end': '2025-12-31',
 'test_start': '2026-01-01',
 'test_end': '2026-09-30',
 'train_backtest_start': '2024-02-08',
 'eligible_codes': ['159608',
  '159703',
  '159713',
  '159715',
  '159745',
  '159761',
  '159763',
  '159870',
  '159871',
  '159880',
  '159881',
  '159944',
  '510410',
  '512400',
  '515210',
  '516020',
  '516120',
  '516150',
  '516220',
  '516360',
  '516650',
  '516710',
  '516750',
  '516780',
  '516890',
  '561800',
  '562800'],
 'train_sharpe': 2.9483040382238896,
 'train_final_nav': 1.1817458116064528,
 'valid_sharpe': 1.3824100344105898,
 'valid_final_nav': 1.0836305696934991,
 'train_max_d